# ETAPA 5 — Conversão de SQL para Pandas

Para inserir os arquivos, basta seguir as etapas abaixo:



1.   Execute o bloco de código abaixo;
2.   No campo Upload de arquivos, selecione a opção "Escolher arquivos";
3.   Adicione os arquivos CSV.

Após isso, é necessário executar o seguinte código antes de executar as consultas: "Ler as TABELAS → DataFrames do Pandas/Preparação para Consultas"

Com a conclusão dessas etapas, será possível executar a consulta desejada.

Visto que alguns requisitos continham apenas inserções ou atualizações de dados, neste notebook estão contidas apenas as consultas das informações.

In [1]:
import pandas as pd
from google.colab import files
print("Upload dos arquivos")
uploaded = files.upload()

# Configurações comuns de leitura
SEP = ';'
ENC_LIST = ['utf-8', 'latin-1']  # tenta utf-8 e, se falhar, latin-1

def ler_csv_flex(nome):
    for enc in ENC_LIST:
        try:
            return pd.read_csv(nome, sep=SEP, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(nome, sep=SEP)


Upload dos arquivos


Saving estoque.csv to estoque.csv
Saving venda.csv to venda.csv
Saving produto.csv to produto.csv
Saving item_venda.csv to item_venda.csv
Saving funcionario.csv to funcionario.csv
Saving fornecedor_produto.csv to fornecedor_produto.csv
Saving fornecedor.csv to fornecedor.csv
Saving cliente.csv to cliente.csv
Saving categoria.csv to categoria.csv


Ler as TABELAS → DataFrames do Pandas/Preparação para Consultas

In [2]:
categoria          = ler_csv_flex('categoria.csv')
fornecedor         = ler_csv_flex('fornecedor.csv')
fornecedor_produto = ler_csv_flex('fornecedor_produto.csv')
produto            = ler_csv_flex('produto.csv')
estoque            = ler_csv_flex('estoque.csv')
cliente            = ler_csv_flex('cliente.csv')
funcionario        = ler_csv_flex('funcionario.csv')
venda              = ler_csv_flex('venda.csv')
item_venda         = ler_csv_flex('item_venda.csv')

# Converter data de venda
if 'data_venda' in venda.columns:
    venda['data_venda'] = pd.to_datetime(venda['data_venda'], errors='coerce')

# Garantir que ESTOQUE não tenha a PK como índice
estoque = estoque.reset_index(drop=True)

# cópias para não bagunçar os originais (se precisar comparar depois)
produto_n   = produto.copy()
estoque_n   = estoque.copy()
categoria_n = categoria.copy()
cliente_n   = cliente.copy()
funcionario_n = funcionario.copy()
venda_n     = venda.copy()
item_venda_n  = item_venda.copy()
fornecedor_n  = fornecedor.copy()
forn_prod_n   = fornecedor_produto.copy()

# 1) datas
venda_n['data_venda'] = pd.to_datetime(venda_n['data_venda'], errors='coerce')

# 2) garantir que ESTOQUE não use a PK como índice e renomear a PK
estoque_n = estoque_n.reset_index(drop=True).rename(columns={'idproduto':'idproduto_estoque'})

# 3) desambiguar preco_unitario de item_venda vs produto
item_venda_n = item_venda_n.rename(columns={
    'preco_unitario': 'preco_unitario_item',
    'idItem_venda': 'iditem_venda'  # padroniza lower
})

# Helper para mostrar resultados

def show(df, titulo=None, n=10):
    if titulo:
        print(f"\n### {titulo}")
    display(df.head(n))
    print(f"→ {len(df)} linha(s)")

# CONSULTAS EM PANDAS

Q009 — Cálculo automático do valor total da venda

In [7]:
iv_totais = (
    item_venda_n
        .assign(valor_item=lambda d: d['quantidade'] * d['preco_unitario_item'])
        .groupby('venda_idvenda', as_index=False)
        .agg(valor_total_calculado=('valor_item', 'sum'))
)
q009 = (
    venda_n.merge(iv_totais, left_on='idvenda', right_on='venda_idvenda', how='left')
           .merge(cliente_n, left_on='cliente_idcliente', right_on='idcliente', how='left')
           .loc[:, ['idvenda', 'data_venda', 'valor_total_calculado',
                    'idcliente', 'nome']]
           .rename(columns={'nome': 'nome_cliente'})
           .sort_values(['data_venda', 'idvenda'], ascending=[True, True])
)

show(q009, "Q009 — Cálculo automático do valor total da venda")


### Q009 — Cálculo automático do valor total da venda


,idvenda,data_venda,valor_total_calculado,idcliente,nome_cliente
0,1,2025-01-05,40.16,1,Maria de Fátima Silva
1,2,2025-01-05,94.60,5,Beatriz Oliveira Lima
2,3,2025-01-06,66.99,10,Luís Felipe Ribeiro
3,4,2025-01-07,13.00,2,João Pedro Santos
4,5,2025-01-08,55.28,8,Guilherme Rocha Ferreira
5,6,2025-01-09,14.00,15,Vanessa Cruz Freitas
6,7,2025-01-10,93.70,20,Erica Pires Mendes
7,8,2025-01-11,33.40,3,Ana Paula Costa
8,9,2025-01-12,103.20,12,Roberto Cavalcanti Teles
9,10,2025-01-15,50.60,4,Carlos Alberto Souza


→ 50 linha(s)


Q010 — Produtos com estoque abaixo do mínimo (Versão com subconsulta)

In [6]:
q010 = (
    produto_n.merge(estoque_n, left_on='estoque_idproduto', right_on='idproduto_estoque', how='inner')
             .merge(categoria_n, left_on='categoria_idcategoria', right_on='idcategoria', how='left')
             .assign(quantidade_faltante=lambda d: d['qtd_minima'] - d['qtd_atual'])
             .query('qtd_atual < qtd_minima')
             .sort_values(['quantidade_faltante','nome_produto'], ascending=[False, True])
             .loc[:, ['idproduto','nome_produto','nome','preco_unitario','qtd_atual','qtd_minima','quantidade_faltante']]
             .rename(columns={'nome':'categoria'})
)
show(q010, "Q010 — Produtos com estoque abaixo do mínimo")


### Q010 — Produtos com estoque abaixo do mínimo


,idproduto,nome_produto,categoria,preco_unitario,qtd_atual,qtd_minima,quantidade_faltante
13,14,Filé de Tilápia Congelado 500g,Carnes e Aves,35.9,1,30,29
10,11,Acém Bovino (Kg),Carnes e Aves,29.9,40,50,10
26,27,Pizza Muçarela Congelada,Congelados,28.5,23,25,2


→ 3 linha(s)


Q011 — Informações das vendas por período

In [5]:
venda_n['periodo'] = venda_n['data_venda'].dt.to_period('M').astype(str)
q011 = (
    venda_n.groupby('periodo', as_index=False)
           .agg(
               quantidade_vendas=('idvenda','count'),
               valor_total_vendido=('valor_total','sum'),
               valor_medio_venda=('valor_total','mean'),
               maior_venda=('valor_total','max'),
               menor_venda=('valor_total','min')
           )
           .sort_values('periodo', ascending=False)
)
show(q011, "Q011 — Informações das vendas por período")


### Q011 — Informações das vendas por período


,periodo,quantidade_vendas,valor_total_vendido,valor_medio_venda,maior_venda,menor_venda
8,2025-09,2,190.99,95.495,180.00,10.99
7,2025-08,4,205.70,51.425,80.60,19.80
6,2025-07,4,257.00,64.250,125.40,6.20
5,2025-06,5,264.70,52.940,105.15,3.75
4,2025-05,5,247.59,49.518,92.10,11.50
3,2025-04,5,385.14,77.028,160.30,5.99
2,2025-03,5,341.50,68.300,145.20,20.00
1,2025-02,10,720.73,72.073,190.10,10.99
0,2025-01,10,645.58,64.558,120.50,15.00


→ 9 linha(s)


Q012 — Produtos mais vendidos

In [4]:
iv_prod = (
    item_venda_n
      .merge(produto_n,   left_on='produto_idproduto', right_on='idproduto', how='inner')
      .merge(categoria_n, left_on='categoria_idcategoria', right_on='idcategoria', how='left')
)
iv_prod['valor_item'] = iv_prod['quantidade'] * iv_prod['preco_unitario_item']

q012 = (
    iv_prod.groupby(['idproduto','nome_produto','nome','preco_unitario'], as_index=False)
           .agg(
               quantidade_vendida     = ('quantidade','sum'),
               numero_vendas          = ('venda_idvenda','count'),
               valor_total_vendido    = ('valor_item','sum'),
               preco_medio            = ('preco_unitario','mean'),
               maior_quantidade_venda = ('quantidade','max')
           )
           .query('quantidade_vendida > 5')
           .sort_values(['quantidade_vendida','valor_total_vendido'], ascending=[False, False])
           .rename(columns={'nome':'categoria'})
)
show(q012, "Q012 — Produtos mais vendidos")


### Q012 — Produtos mais vendidos


,idproduto,nome_produto,categoria,preco_unitario,quantidade_vendida,numero_vendas,valor_total_vendido,preco_medio,maior_quantidade_venda
23,24,"Água Mineral sem Gás 1,5L",Bebidas,2.50,30,2,75.00,2.50,20
21,22,Cerveja Pilsen Lata 350ml,Bebidas,3.49,20,2,69.80,3.49,10
35,36,Detergente Líquido 500ml,Limpeza Doméstica,2.80,16,2,44.80,2.80,11
36,37,Água Sanitária 1L,Limpeza Doméstica,4.90,11,3,53.90,4.90,5
17,18,Açúcar Refinado 1kg,Mercearia Básica,4.20,11,2,46.20,4.20,6
19,20,Macarrão Espaguete 500g,Mercearia Básica,3.75,11,3,41.25,3.75,5
46,47,Pacote de Balas Sortidas 500g,Doces e Guloseimas,11.90,10,2,119.00,11.90,5
40,41,Pão Francês (Unidade),Padaria,0.80,10,2,8.00,0.80,5
48,49,Chiclete Bola Tutti Frutti,Doces e Guloseimas,0.50,10,1,5.00,0.50,10
27,28,Batata Palito Congelada 400g,Congelados,12.99,9,2,116.91,12.99,5


→ 19 linha(s)


Q013 — Clientes que mais compraram

In [3]:
met_vendas_cli = (
    venda_n.groupby('cliente_idcliente', as_index=False)
           .agg(
               total_compras=('idvenda','nunique'),
               valor_total_gasto=('valor_total','sum'),
               ticket_medio=('valor_total','mean'),
               maior_compra=('valor_total','max'),
               menor_compra=('valor_total','min')
           )
)

itens_por_cli = (
    venda_n.merge(item_venda_n, left_on='idvenda', right_on='venda_idvenda', how='left')
           .groupby('cliente_idcliente', as_index=False)
           .agg(quantidade_total_itens=('quantidade','sum'))
)

q013 = (
    cliente_n.merge(met_vendas_cli, left_on='idcliente', right_on='cliente_idcliente', how='left')
             .merge(itens_por_cli,   left_on='idcliente', right_on='cliente_idcliente', how='left')
             .fillna({'total_compras':0, 'valor_total_gasto':0, 'ticket_medio':0,
                      'maior_compra':0, 'menor_compra':0, 'quantidade_total_itens':0})
             .query('total_compras > 0')
             .loc[:, ['idcliente','nome','email','cpf','telefone',
                      'total_compras','valor_total_gasto','ticket_medio',
                      'maior_compra','menor_compra','quantidade_total_itens']]
             .sort_values(['total_compras','valor_total_gasto','nome'], ascending=[False, False, True])
)

show(q013, "Q013 — Clientes que mais compraram (ordenado por quantidade de compras)")



### Q013 — Clientes que mais compraram (ordenado por quantidade de compras)


,idcliente,nome,email,cpf,telefone,total_compras,valor_total_gasto,ticket_medio,maior_compra,menor_compra,quantidade_total_itens
0,1,Maria de Fátima Silva,maria.silva@email.com,12345678901,(11) 98765-1234,3.0,115.37,38.456667,53.40,15.99,10.0
17,18,Cintia Regina Santos,cintia.santos@email.com,87654321098,(81) 93334-3536,2.0,370.10,185.050000,190.10,180.00,12.0
9,10,Luís Felipe Ribeiro,luis.ribeiro@email.com,1234567890,(11) 91011-2131,2.0,183.40,91.700000,105.15,78.25,15.0
49,50,Ingrid Mendes Silva,ingrid.silva@email.com,9385776240,(11) 92930-3132,2.0,176.30,88.150000,95.70,80.60,20.0
19,20,Erica Pires Mendes,erica.mendes@email.com,9876543210,(11) 93940-4142,2.0,172.80,86.400000,98.30,74.50,17.0
4,5,Beatriz Oliveira Lima,beatriz.lima@email.com,56789012345,(51) 95566-7788,2.0,148.40,74.200000,120.50,27.90,9.0
29,30,Otávio Morais Duarte,otavio.duarte@email.com,9182736450,(11) 96970-7172,2.0,140.15,70.075000,99.90,40.25,10.0
44,45,Diego Santos Lima,diego.lima@email.com,54830221795,(51) 91415-1617,2.0,135.40,67.700000,115.60,19.80,10.0
39,40,Yara Pereira Mendes,yara.mendes@email.com,9284765130,(11) 99900-0102,2.0,131.39,65.695000,125.40,5.99,7.0
7,8,Guilherme Rocha Ferreira,guilherme.rocha@email.com,89012345678,(81) 92233-4455,2.0,126.00,63.000000,70.30,55.70,12.0


→ 32 linha(s)


Q014 — Busca de clientes por nome ou CPF

In [ ]:
q014 = (
    cliente_n[
        cliente_n['nome'].str.contains('Silva', case=False, na=False) |
        cliente_n['cpf'].astype(str).str.startswith('123', na=False)
    ]
    .sort_values('nome')
    .loc[:, ['idcliente','nome','email','cpf','telefone']]
)
show(q014, "Q014 — Busca de clientes por nome ou CPF")


### Q014 — Busca de clientes por nome ou CPF


,idcliente,nome,email,cpf,telefone
22,23,Hugo Mendes Silva,hugo.silva@email.com,32415069783,(31) 94849-5051
49,50,Ingrid Mendes Silva,ingrid.silva@email.com,9385776240,(11) 92930-3132
9,10,Luís Felipe Ribeiro,luis.ribeiro@email.com,1234567890,(11) 91011-2131
0,1,Maria de Fátima Silva,maria.silva@email.com,12345678901,(11) 98765-1234
12,13,Silvana Xavier Duarte,silvana.duarte@email.com,32109876543,(31) 91819-2021
37,38,Wanda Silva Almeida,wanda.almeida@email.com,87062543918,(81) 99394-9596


→ 6 linha(s)


Q015 — Busca de produtos por nome ou categoria

In [ ]:
q015 = (
    produto_n.merge(categoria_n, left_on='categoria_idcategoria', right_on='idcategoria', how='left')
             .loc[
                 lambda d: d['nome_produto'].str.contains('leite', case=False, na=False) |
                           d['nome'].str.contains('laticínios', case=False, na=False),
                 ['idproduto','nome_produto','preco_unitario','estoque_idproduto','nome']
             ]
             .rename(columns={'nome':'categoria'})
             .sort_values('nome_produto')
)
show(q015, "Q015 — Produtos por nome ou categoria")


### Q015 — Produtos por nome ou categoria


,idproduto,nome_produto,preco_unitario,estoque_idproduto,categoria
45,46,Barra de Chocolate ao Leite 90g,6.50,46,Doces e Guloseimas
49,50,Doce de Leite Pote 400g,14.00,50,Doces e Guloseimas
7,8,Iogurte Natural 170g,2.99,8,Laticínios e Frios
5,6,Leite Integral 1L,4.59,6,Laticínios e Frios
8,9,Manteiga com Sal 200g,12.50,9,Laticínios e Frios
9,10,Presunto Cozido (Kg),25.90,10,Laticínios e Frios
6,7,Queijo Muçarela (Kg),38.90,7,Laticínios e Frios


→ 7 linha(s)


Q016 — Consulta de estoque agrupado por fornecedor

In [ ]:
q016 = (
    fornecedor_n.merge(forn_prod_n, left_on='idfornecedor', right_on='fornecedor_idfornecedor')
                .merge(produto_n,   left_on='produto_idproduto', right_on='idproduto')
                .merge(categoria_n, left_on='categoria_idcategoria', right_on='idcategoria')
                .merge(estoque_n,   left_on='estoque_idproduto',  right_on='idproduto_estoque', how='left')
)

q016 = (q016
        .rename(columns={'nome_x':'nome_fornecedor', 'nome_y':'categoria', 'qtd_atual':'quantidade_estoque'})
        .loc[:, ['idfornecedor','nome_fornecedor','telefone','email','categoria',
                 'nome_produto','preco_unitario','quantidade_estoque']]
        .sort_values(['nome_fornecedor','categoria','nome_produto'])
)
show(q016, "Q016 — Estoque por fornecedor")


### Q016 — Estoque por fornecedor


,idfornecedor,nome_fornecedor,telefone,email,categoria,nome_produto,preco_unitario,quantidade_estoque
20,11,Alimentos Frescos SC,(48) 91213-1415,fresh@alimentossc.com,Hortifrúti,Banana Nanica (Kg),5.99,150
21,11,Alimentos Frescos SC,(48) 91213-1415,fresh@alimentossc.com,Hortifrúti,Batata Inglesa (Kg),7.89,180
43,38,Alvejantes Profissionais,(22) 99394-9596,vendas@alvejantes.com,Limpeza Doméstica,Água Sanitária 1L,4.90,500
39,33,Aves Premium,(83) 97879-8081,premium@aves.com,Carnes e Aves,Peito de Frango (Kg),18.90,320
49,44,Açúcares e Farinhas,(98) 91112-1314,pedidos@acucares.com,Mercearia Básica,Feijão Carioca 1kg,8.50,600
37,30,Balas e Chicletes Vitoria,(69) 96970-7172,contato@balasechicletes.com,Doces e Guloseimas,Chiclete Bola Tutti Frutti,0.50,700
53,49,Biscoitos Salgados SP,(61) 92627-2829,salgados@biscoitos.com,Doces e Guloseimas,Biscoito Recheado Chocolate,3.20,400
44,39,Bolos Prontos Delícia,(28) 99697-9899,bolos@prontos.com,Padaria,Bolo de Fubá Caseiro (Unidade),15.00,100
24,14,Cereais Matinais BR,(85) 92122-2324,vendas@cereaisbr.com,Mercearia Básica,Óleo de Soja 900ml,6.99,400
41,35,Cervejas Artesanais,(16) 98485-8687,cervejas@artesanais.com,Bebidas,Cerveja Pilsen Lata 350ml,3.49,500


→ 55 linha(s)


Q017 — Histórico de compras de um cliente

In [8]:
q017 = (
    cliente_n.merge(venda_n,      left_on='idcliente',         right_on='cliente_idcliente')
             .merge(item_venda_n, left_on='idvenda',           right_on='venda_idvenda')
             .merge(produto_n,    left_on='produto_idproduto', right_on='idproduto')
             .assign(subtotal=lambda d: d['quantidade'] * d['preco_unitario_item'])
             .sort_values(['nome','data_venda','nome_produto'], ascending=[True, False, True])
             .loc[:, ['idcliente','nome','email','cpf','idvenda','data_venda','valor_total',
                      'idproduto','nome_produto','quantidade','preco_unitario_item','subtotal']]
             .rename(columns={'nome':'nome_cliente','preco_unitario_item':'preco_unitario'})
)
show(q017, "Q017 — Histórico de compras do cliente")


### Q017 — Histórico de compras do cliente


,idcliente,nome_cliente,email,cpf,idvenda,data_venda,valor_total,idproduto,nome_produto,quantidade,preco_unitario,subtotal
9,3,Ana Paula Costa,ana.paula@email.com,34567890123,8,2025-01-11,33.45,18,Açúcar Refinado 1kg,5,4.20,21.00
8,3,Ana Paula Costa,ana.paula@email.com,34567890123,8,2025-01-11,33.45,4,Tomate Salada (Kg),2,6.20,12.40
46,16,Andréia Correia Pires,andreia.pires@email.com,65432109876,30,2025-04-20,160.30,11,Acém Bovino (Kg),2,29.90,59.80
45,16,Andréia Correia Pires,andreia.pires@email.com,65432109876,30,2025-04-20,160.30,16,Arroz Agulhinha 5kg,4,24.99,99.96
44,16,Andréia Correia Pires,andreia.pires@email.com,65432109876,30,2025-04-20,160.30,6,Leite Integral 1L,3,4.59,13.77
15,5,Beatriz Oliveira Lima,beatriz.lima@email.com,56789012345,37,2025-06-05,27.90,20,Macarrão Espaguete 500g,5,3.75,18.75
13,5,Beatriz Oliveira Lima,beatriz.lima@email.com,56789012345,2,2025-01-05,120.50,11,Acém Bovino (Kg),3,29.90,89.70
14,5,Beatriz Oliveira Lima,beatriz.lima@email.com,56789012345,2,2025-01-05,120.50,37,Água Sanitária 1L,1,4.90,4.90
12,4,Carlos Alberto Souza,carlos.souza@email.com,45678901234,28,2025-04-10,18.25,42,Bolo de Fubá Caseiro (Unidade),1,15.00,15.00
10,4,Carlos Alberto Souza,carlos.souza@email.com,45678901234,10,2025-01-15,65.50,48,Biscoito Recheado Chocolate,8,3.20,25.60


→ 90 linha(s)


Q018 — Desempenho por funcionário

In [9]:
q018 = (
    funcionario_n.merge(venda_n,      left_on='idfuncionario', right_on='funcionario_idfuncionario')
                 .merge(item_venda_n, left_on='idvenda',       right_on='venda_idvenda')
                 .groupby(['idfuncionario','nome','cargo','cpf'], as_index=False)
                 .agg(
                     total_vendas=('idvenda','count'),
                     valor_total_vendido=('valor_total','sum'),
                     ticket_medio=('valor_total','mean'),
                     maior_venda=('valor_total','max'),
                     menor_venda=('valor_total','min'),
                     quantidade_itens_vendidos=('quantidade','sum'),
                     clientes_atendidos=('cliente_idcliente','nunique')
                 )
                 .sort_values(['valor_total_vendido','total_vendas'], ascending=[False, False])
                 .rename(columns={'nome':'nome_funcionario'})
)
show(q018, "Q018 — Desempenho por funcionário")


### Q018 — Desempenho por funcionário


,idfuncionario,nome_funcionario,cargo,cpf,total_vendas,valor_total_vendido,ticket_medio,maior_venda,menor_venda,quantidade_itens_vendidos,clientes_atendidos
1,4,Jéssica Soares Reis,Caixa - Vespertino,44556677883,42,3698.79,88.066429,190.1,5.99,156,20
0,3,Marcelo Aguiar Neto,Caixa - Matutino,33445566772,48,3347.59,69.741458,145.2,3.75,133,19


→ 2 linha(s)


Q019 — Produtos que atingiram o nível mínimo de estoque

In [10]:
q019 = (
    produto_n.merge(estoque_n,        left_on='estoque_idproduto', right_on='idproduto_estoque', how='left')
             .merge(categoria_n,      left_on='categoria_idcategoria', right_on='idcategoria', how='left')
             .merge(forn_prod_n,      left_on='idproduto', right_on='produto_idproduto', how='left')
             .merge(fornecedor_n,     left_on='fornecedor_idfornecedor', right_on='idfornecedor', how='left')
             .assign(quantidade_faltante=lambda d: d['qtd_minima'] - d['qtd_atual'])
             .query('qtd_atual.notnull() and qtd_atual <= qtd_minima')
             .sort_values(['quantidade_faltante','nome_produto'], ascending=[False, True])
             .loc[:, ['idproduto','nome_produto','nome_x','qtd_atual','qtd_minima',
                      'quantidade_faltante','idfornecedor','nome_y','telefone','email','preco_unitario']]
             .rename(columns={'nome_x':'categoria','nome_y':'nome_fornecedor'})
)
show(q019, "Q019 — Produtos no nível mínimo de estoque")



### Q019 — Produtos no nível mínimo de estoque


,idproduto,nome_produto,categoria,qtd_atual,qtd_minima,quantidade_faltante,idfornecedor,nome_fornecedor,telefone,email,preco_unitario
15,14,Filé de Tilápia Congelado 500g,Carnes e Aves,1,30,29,23,Peixes e Frutos do Mar,(84) 94849-5051,frutosdomar@peixes.com,35.9
12,11,Acém Bovino (Kg),Carnes e Aves,40,50,10,3,Frigorífico Boi Forte,(21) 99876-5432,comercial@boiforte.com,29.9
30,27,Pizza Muçarela Congelada,Congelados,23,25,2,26,Pizzas Congeladas Top,(79) 95758-5960,top@pizzascongeladas.com,28.5


→ 3 linha(s)


Q020 — Faturamento por categoria

In [11]:
iv_full = (
    categoria_n.merge(produto_n,   left_on='idcategoria',          right_on='categoria_idcategoria')
               .merge(item_venda_n, left_on='idproduto',           right_on='produto_idproduto')
)
iv_full['valor_item'] = iv_full['quantidade'] * iv_full['preco_unitario_item']

q020 = (
    iv_full.groupby(['idcategoria','nome'], as_index=False)
           .agg(
               total_produtos=('idproduto','nunique'),
               total_vendas=('venda_idvenda','nunique'),
               quantidade_vendida=('quantidade','sum'),
               valor_total_vendido=('valor_item','sum'),
               preco_medio=('preco_unitario_item','mean'),
               maior_preco=('preco_unitario_item','max'),
               menor_preco=('preco_unitario_item','min'),
               quantidade_media_por_venda=('quantidade','mean')
           )
           .assign(ticket_medio_categoria=lambda d: (d['valor_total_vendido'] / d['total_vendas']).round(2))
           .query('total_vendas > 0')
           .sort_values(['valor_total_vendido','quantidade_vendida'], ascending=[False, False])
           .rename(columns={'nome':'nome_categoria'})
)
show(q020, "Q020 — Faturamento por categoria")


### Q020 — Faturamento por categoria


,idcategoria,nome_categoria,total_produtos,total_vendas,quantidade_vendida,valor_total_vendido,preco_medio,maior_preco,menor_preco,quantidade_media_por_venda,ticket_medio_categoria
2,3,Carnes e Aves,5,10,22,508.60,21.860000,35.90,10.90,2.200000,50.86
3,4,Mercearia Básica,5,11,38,347.82,12.281818,24.99,3.75,3.454545,31.62
1,2,Laticínios e Frios,5,8,26,297.93,14.995000,38.90,2.99,3.250000,37.24
6,7,Higiene Pessoal,5,10,19,291.79,14.789000,22.90,4.50,1.900000,29.18
5,6,Congelados,5,7,16,281.61,20.468571,28.50,12.99,2.285714,40.23
7,8,Limpeza Doméstica,5,11,37,238.77,10.180909,18.90,2.80,3.363636,21.71
9,10,Doces e Guloseimas,5,6,34,196.10,8.000000,14.00,0.50,5.666667,32.68
4,5,Bebidas,5,7,54,187.69,6.858750,14.90,2.50,6.750000,26.81
0,1,Hortifrúti,5,9,21,144.50,6.606667,9.99,3.50,2.333333,16.06
8,9,Padaria,5,10,22,115.00,7.900000,15.00,0.80,2.200000,11.50


→ 10 linha(s)


Q011 (extra) — Ordenar vendas por data

In [12]:
vendas_itens = (
    venda_n.merge(cliente_n,    left_on='cliente_idcliente',      right_on='idcliente')
           .merge(funcionario_n, left_on='funcionario_idfuncionario', right_on='idfuncionario')
           .merge(item_venda_n,  left_on='idvenda',                right_on='venda_idvenda', how='left')
)

q011_ord = (
    vendas_itens.groupby(['idvenda','data_venda','valor_total',
                          'idcliente','nome_x','email',
                          'idfuncionario','nome_y','cargo'], as_index=False)
                .agg(
                    quantidade_itens=('iditem_venda','count'),
                    quantidade_total_produtos=('quantidade','sum')
                )
                .sort_values('data_venda', ascending=False)
                .rename(columns={'nome_x':'nome_cliente','nome_y':'nome_funcionario'})
)
q011_ord['quantidade_total_produtos'] = q011_ord['quantidade_total_produtos'].fillna(0)
show(q011_ord, "Q011 — Ordenar vendas por data (com contagem de itens)")


### Q011 — Ordenar vendas por data (com contagem de itens)


,idvenda,data_venda,valor_total,idcliente,nome_cliente,email,idfuncionario,nome_funcionario,cargo,quantidade_itens,quantidade_total_produtos
49,50,2025-09-05,180.00,18,Cintia Regina Santos,cintia.santos@email.com,4,Jéssica Soares Reis,Caixa - Vespertino,2,7
48,49,2025-09-01,10.99,12,Roberto Cavalcanti Teles,roberto.teles@email.com,3,Marcelo Aguiar Neto,Caixa - Matutino,1,1
47,48,2025-08-15,70.30,8,Guilherme Rocha Ferreira,guilherme.rocha@email.com,4,Jéssica Soares Reis,Caixa - Vespertino,2,7
46,47,2025-08-10,35.00,2,João Pedro Santos,joao.santos@email.com,3,Marcelo Aguiar Neto,Caixa - Matutino,1,1
45,46,2025-08-05,80.60,50,Ingrid Mendes Silva,ingrid.silva@email.com,4,Jéssica Soares Reis,Caixa - Vespertino,2,15
44,45,2025-08-01,19.80,45,Diego Santos Lima,diego.lima@email.com,3,Marcelo Aguiar Neto,Caixa - Matutino,1,3
43,44,2025-07-15,125.40,40,Yara Pereira Mendes,yara.mendes@email.com,4,Jéssica Soares Reis,Caixa - Vespertino,1,5
42,43,2025-07-10,6.20,35,Túlio Freitas Lima,tulio.lima@email.com,3,Marcelo Aguiar Neto,Caixa - Matutino,1,1
41,42,2025-07-05,99.90,30,Otávio Morais Duarte,otavio.duarte@email.com,4,Jéssica Soares Reis,Caixa - Vespertino,2,5
40,41,2025-07-01,25.50,25,Júlio César Freitas,julio.freitas@email.com,3,Marcelo Aguiar Neto,Caixa - Matutino,2,2


→ 50 linha(s)
